# Task 4.0 (phần tối thiểu): tạo dữ liệu tensor đã xử lý

Chuyển graph JSON của Task 3.3 thành tensor theo `docs/4_0_data_interface_spec.md` (v1.1).

**Chuẩn bị:** Add Input dataset `DiverseVul 9077 - Joern Code Graphs` (chứa `graph_a`, `graph_b`). Chỉ cần CPU.

**Sau khi chạy xong:** Save Version (Save & Run All) rồi New Dataset từ output, đặt tên `DiverseVul 9077 - Processed Graph Data`. Output chỉ gồm thư mục `processed`.

In [ ]:
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True, capture_output=True).stdout)

## Tìm thư mục chứa graph_a và graph_b

In [ ]:
roots = [p.parent for p in Path("/kaggle/input").rglob("graph_a") if p.is_dir() and (p.parent / "graph_b").is_dir()]
assert roots, "Không thấy graph_a và graph_b, hãy Add Input dataset DiverseVul 9077 - Joern Code Graphs."
GRAPH_ROOT = roots[0]
OUT_DIR = Path("/kaggle/working/processed")
print("graph root:", GRAPH_ROOT)
print("graph_a:", sum(1 for _ in (GRAPH_ROOT / "graph_a").glob("*.json")), "tệp;",
      "graph_b:", sum(1 for _ in (GRAPH_ROOT / "graph_b").glob("*.json")), "tệp")

## Tạo dữ liệu

In [ ]:
def run(cmd):
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"Lệnh thất bại với mã {proc.returncode}")

run([sys.executable, "scripts/build_processed_data.py", "--graph-root", str(GRAPH_ROOT), "--out-dir", str(OUT_DIR)])

## Kiểm tra đầu ra

In [ ]:
import json
import torch

meta = json.loads((OUT_DIR / "metadata.json").read_text(encoding="utf-8"))
print("Mẫu hợp lệ:", meta["n_valid_samples"], "| bị bỏ:", meta["skipped"])
for protocol, info in meta["protocols"].items():
    print(protocol, "K =", info["k"], "| P95 thô =", info["train_token_len_p95_raw"],
          "| vocab token =", info["n_tokens"], "| vocab loại node =", info["n_node_types"])
    for name, s in info["splits"].items():
        print("   ", name, s)
sample = torch.load(OUT_DIR / "project_wise" / "graph_b_train.pt")[0]
print({k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in sample.items()})
print(sorted(p.name for p in OUT_DIR.rglob("*") if p.is_file()))